# 01 — Dataset Exploration and Understanding

This notebook explores the official SQuAD 2.0 dataset for the project **Knowing When Not to Answer: Abstention in Retrieval-Augmented Question Answering**.

The proposal calls for a manageable, balanced subset of approximately 6,000–10,000 examples, retaining passage identifiers and source metadata, and splitting by source article or passage group where feasible.

**Important label definition:** SQuAD's `is_impossible` label refers to whether the answer is present in that question's *associated passage*. It is not a direct annotation of whether arbitrary retrieved top-k evidence supports an answer. We will preserve this distinction throughout the experiments.

## 1. Imports and reproducibility

We use Pandas for tables, NumPy for numerical operations, and Matplotlib/Seaborn for visual inspection. A fixed seed makes random sampling reproducible.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 160)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
SPLITS = ROOT / "data" / "splits"
for p in [RAW, PROCESSED, SPLITS]:
    p.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT.resolve())

## 2. Download official SQuAD 2.0 files

The notebook downloads the official train and development JSON files only when they are not already present. The data source is the [official SQuAD website](https://rajpurkar.github.io/SQuAD-explorer/). Review its terms and source-content conditions before redistributing any data.

In [ ]:
from urllib.request import urlretrieve

URLS = {
    "train-v2.0.json": "https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v2.0.json",
    "dev-v2.0.json": "https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v2.0.json",
}
for name, url in URLS.items():
    dest = RAW / name
    if dest.exists() and dest.stat().st_size > 0:
        print(f"[EXISTS] {name}: {dest.stat().st_size:,} bytes")
    else:
        print("[DOWNLOAD]", name)
        urlretrieve(url, dest)
        print("[SAVED]", dest)

## 3. Load JSON and inspect its hierarchy

SQuAD is nested as **articles → paragraphs → question-answer records (`qas`)**. A paragraph is the context passage associated with one or more questions.

In [ ]:
with (RAW / "train-v2.0.json").open(encoding="utf-8") as f:
    train_json = json.load(f)
with (RAW / "dev-v2.0.json").open(encoding="utf-8") as f:
    dev_json = json.load(f)

print("Top-level keys:", train_json.keys())
print("Train articles:", len(train_json["data"]))
print("Dev articles:", len(dev_json["data"]))

article = train_json["data"][0]
paragraph = article["paragraphs"][0]
qa = paragraph["qas"][0]
print("\nArticle title:", article["title"])
print("Article keys:", article.keys())
print("Paragraph keys:", paragraph.keys())
print("Context preview:", paragraph["context"][:500])
print("QA keys:", qa.keys())
print("Question:", qa["question"])
print("Question ID:", qa["id"])
print("is_impossible:", qa["is_impossible"])
print("Answers:", qa["answers"])

## 4. Flatten the nested structure

We create one row per question. We preserve the original question ID, article title, source split, context and all answer annotations. `paragraph_id` is a reproducible ID built from source split and article/paragraph positions; it is not an original SQuAD field.

`is_impossible=False` means an answer is annotated in the associated passage. It does **not** automatically mean a future retrieval system will retrieve sufficient evidence.

In [ ]:
def flatten_squad(obj, source_split):
    rows = []
    for article_i, article in enumerate(obj["data"]):
        for paragraph_i, paragraph in enumerate(article["paragraphs"]):
            pid = f"{source_split}_a{article_i}_p{paragraph_i}"
            for qa in paragraph["qas"]:
                answers = qa.get("answers", [])
                impossible = bool(qa["is_impossible"])
                first = answers[0] if answers else {}
                rows.append({
                    "source_split": source_split,
                    "article_title": article["title"],
                    "paragraph_id": pid,
                    "question_id": qa["id"],
                    "context": paragraph["context"],
                    "question": qa["question"],
                    "is_impossible": impossible,
                    "is_answerable": not impossible,
                    "answers": answers,
                    "answer_text": first.get("text"),
                    "answer_start": first.get("answer_start"),
                })
    return pd.DataFrame(rows)

train_df = flatten_squad(train_json, "train")
dev_df = flatten_squad(dev_json, "dev")
print("Train:", train_df.shape, "| Dev:", dev_df.shape)
display(train_df.head(3))

## 5. Class distribution

This is the original class distribution. Do not rebalance until the raw dataset has been inspected. Class balance is useful for controlled experiments, but sampling must happen after grouping to avoid article leakage.

In [ ]:
def class_counts(df, name):
    counts = df["is_answerable"].value_counts().reindex([True, False], fill_value=0)
    return pd.DataFrame({
        "split": name,
        "class": ["Answerable", "Unanswerable"],
        "count": [int(counts[True]), int(counts[False])],
        "percentage": [100*counts[True]/len(df), 100*counts[False]/len(df)]
    })

class_stats = pd.concat([class_counts(train_df, "train"), class_counts(dev_df, "dev")], ignore_index=True)
display(class_stats)
plt.figure(figsize=(8, 5))
sns.barplot(data=class_stats, x="split", y="count", hue="class")
plt.title("SQuAD 2.0 class distribution")
plt.tight_layout()
plt.show()

## 6. Inspect examples and data quality

Read several examples from both classes. In SQuAD 2.0, unanswerable means the associated passage does not provide an answer; it does not mean the question is impossible to answer from all knowledge sources.

In [ ]:
def show_examples(df, answerable, n=3, seed=SEED):
    sample = df[df.is_answerable == answerable].sample(
        n=min(n, (df.is_answerable == answerable).sum()), random_state=seed
    )
    for _, r in sample.iterrows():
        print("="*90)
        print("Title:", r.article_title)
        print("Question ID:", r.question_id)
        print("Question:", r.question)
        print("Answerable in paired passage:", r.is_answerable)
        print("Reference answer:", r.answer_text)
        print("Context:", r.context[:700], "\n")

show_examples(train_df, True)
show_examples(train_df, False)

In [ ]:
def quality_report(df, split):
    return {
        "split": split,
        "rows": len(df),
        "missing_question": int(df.question.isna().sum()),
        "empty_question": int(df.question.fillna("").str.strip().eq("").sum()),
        "missing_context": int(df.context.isna().sum()),
        "empty_context": int(df.context.fillna("").str.strip().eq("").sum()),
        "duplicate_question_ids": int(df.question_id.duplicated().sum()),
        "duplicate_question_context_pairs": int(df.duplicated(["question", "context"]).sum()),
    }

display(pd.DataFrame([quality_report(train_df, "train"), quality_report(dev_df, "dev")]))

## 7. Validate answer offsets

For an answerable example, the annotated answer should match the context at `answer_start`. We test the first annotation and report the fraction that matches. Keep the complete `answers` list for later evaluation.

In [ ]:
def offset_valid(row):
    if not row.is_answerable:
        return None
    if not isinstance(row.answer_text, str) or pd.isna(row.answer_start):
        return False
    start = int(row.answer_start)
    return row.context[start:start+len(row.answer_text)] == row.answer_text

for name, df in [("train", train_df), ("dev", dev_df)]:
    valid = df.apply(offset_valid, axis=1)
    answerable_valid = valid.dropna()
    print(name, "valid first-answer offsets:",
          f"{answerable_valid.mean():.4%}", f"({answerable_valid.sum()}/{len(answerable_valid)})")

## 8. Passage statistics and article overlap

The number of questions is larger than the number of passages because one paragraph can have several questions. We inspect article overlap between the official files; later we will create our own group-based train/validation/test split from the combined pool.

In [ ]:
def passage_stats(df, split):
    passages = df.drop_duplicates("paragraph_id")
    return {
        "split": split,
        "questions": len(df),
        "articles": df.article_title.nunique(),
        "passages": len(passages),
        "mean_context_chars": passages.context.str.len().mean(),
        "median_context_chars": passages.context.str.len().median(),
        "mean_questions_per_passage": len(df)/max(len(passages), 1),
    }

display(pd.DataFrame([passage_stats(train_df, "train"), passage_stats(dev_df, "dev")]))
overlap = set(train_df.article_title.unique()) & set(dev_df.article_title.unique())
print("Article-title overlap:", len(overlap))
print("Examples:", list(sorted(overlap))[:10])

In [ ]:
unique_train_passages = train_df.drop_duplicates("paragraph_id").copy()
unique_train_passages["context_chars"] = unique_train_passages.context.str.len()

plt.figure(figsize=(9, 5))
sns.histplot(unique_train_passages.context_chars, bins=40, kde=True)
plt.title("Training passage length (characters)")
plt.xlabel("Characters")
plt.tight_layout()
plt.show()
display(unique_train_passages.context_chars.describe())

## 9. Save flattened exploration data

CSV files are easy to inspect. The nested answer annotations are serialized as JSON strings in the CSV. We also attempt Parquet output; install `pyarrow` if it is unavailable.

In [ ]:
def save_exploration(df, name):
    out = df.copy()
    out["answers"] = out["answers"].apply(lambda x: json.dumps(x, ensure_ascii=False))
    out.to_csv(PROCESSED / f"{name}_exploration.csv", index=False, encoding="utf-8")
    print("Saved", PROCESSED / f"{name}_exploration.csv")
    try:
        df.to_parquet(PROCESSED / f"{name}_exploration.parquet", index=False)
        print("Saved Parquet")
    except ImportError:
        print("Parquet skipped: pip install pyarrow")

save_exploration(train_df, "train")
save_exploration(dev_df, "dev")

## 10. Knowledge check

1. What is the difference between an article, paragraph and question?
2. What does `is_impossible=True` mean?
3. Why preserve `question_id` and `paragraph_id`?
4. Why can multiple questions share one context?
5. Why is article-level grouping useful?
6. Why is SQuAD's paired-passage label only a proxy for answerability from a retrieved top-k set?

**Completion criterion:** both source files load, flattening works, class distributions and quality checks are inspected, and exploration files are saved.

**Next:** `02_data_preprocessing.ipynb`.